# Pipeline de Churn em Wealth Management — walkthrough CRISP-DM (v2)

> **Dado 100% sintético.** Este notebook narra o pipeline **v2** (early-warning comportamental, ADR-0001 / ADR-0002) chamando as funções reais de `src/` — o mesmo código que `pipeline.py` orquestra. Não há uma segunda cópia da lógica de geração aqui: se `src/` mudar, este notebook acompanha.

## Contexto de negócio

A carteira simulada é uma **gestora de wealth** (private banking / multi-family office): ~1.200 grupos econômicos, ~R$ 76 bi de AuC, ~50 assessores. Duas perguntas separadas:

- **Direção A — churn do cliente:** quais relações mostram sinais *comportamentais* de deterioração (dias sem contato, queda de cadência, latência de resposta) **antes** da queda de AuC? Prever churn só pela queda de saldo chega tarde.
- **Direção B — advisor attrition:** quanto AuC de uma carteira tende a migrar se o assessor sair da firma? Produto descritivo, **não** entra no classificador (importância medida de 1,3%, correlação com churn individual −0,04).

Não há estimativa de ROI ou de retenção: o dado é sintético e as métricas medem a coerência do gerador e do pipeline, não desempenho em produção.

| Fase CRISP-DM | O que este notebook faz |
|---|---|
| 0 · Setup | imports de `src/`, parâmetros de `conf/base/parameters.yml` |
| 1 · Business Understanding | as duas perguntas, a linguagem ubíqua, as armadilhas |
| 2 · Data Understanding | geração sintética por segmento + auditoria de escala + t-testes |
| 3 · Data Preparation | split estratificado, feature engineering sem leakage |
| 4 · Modeling | baseline v1 reativa vs. v2 early-warning, no mesmo split |
| 5 · Evaluation | CV, IC bootstrap da diferença, importância v2, thresholds por custo |
| 6 · Deployment | scoring ao vivo com o `gb_pipeline_v2.pkl` que o `pipeline.py` gera |

## Fase 0 — Setup

Importamos as funções reais do projeto e os parâmetros versionados. Nada aqui é reescrito: o notebook é uma narração do `pipeline.py`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # rodando a partir de notebooks/

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats

from src.kedro_runner import load_parameters
from src.data_processing.nodes import (
    generate_synthetic_data, generate_advisors_data,
    attach_advisor_and_behavioral_features, inject_data_quality_issues,
    clean_clientes_v2_bruto, aggregate_carteira_exposta_por_assessor,
    split_data, run_feature_engineering,
)
from src.model_training.nodes import (
    benchmark_algorithms, train_and_compare_v1_v2, bootstrap_ic_diferenca_recall,
    cross_validate_v2, train_final_model_v2, get_feature_importance_v2,
    calibrate_thresholds_v2,
)

parameters = load_parameters("../conf/base/parameters.yml")
SEED = parameters.get("random_state", 42)
print("[OK] Fase 0 — funcoes de src/ importadas")
print(f"     n_samples={parameters['n_samples']} | n_advisors={parameters['n_advisors']} | seed={SEED}")
print("     Este notebook NAO escreve em output/ — o pipeline.py e a fonte de verdade dos artefatos.")

[OK] Fase 0 — funcoes de src/ importadas
     n_samples=1200 | n_advisors=50 | seed=42
     Este notebook NAO escreve em output/ — o pipeline.py e a fonte de verdade dos artefatos.


## Fase 1 — Business Understanding

| Pergunta CRISP-DM | Resposta do projeto |
|---|---|
| Qual a dor? | Churn detectado só pela queda de AuC é reativo — a decisão do cliente já aconteceu. |
| Sinal que antecede? | Comportamento de relacionamento: recência de contato, variação de cadência, latência de resposta. |
| Segunda dor, causa distinta? | Advisor attrition — o assessor troca de firma e leva carteira. Não é churn do cliente. |
| Meta analítica? | v2 (early-warning) supera a baseline v1 (reativa) em recall no **mesmo split**, sem declarar ganho de negócio a partir de dado sintético. |
| Como é consumido? | API FastAPI (`api.py`), dashboard (`app.py`), monitor de drift (`monitor.py`), agente (`agent.py`). |

**Armadilhas mapeadas:** (1) *leakage* — features não podem enxergar o encerramento; (2) desbalanceamento (~12% churn) — F1-macro e recall da classe positiva, não acurácia; (3) *training-serving skew* — a transformação vive num lugar só (`sklearn.Pipeline` serializado); (4) *artefato de simulação* — o gerador injeta correlação **parcial e ruidosa** de propósito, com uma feature red-herring, para o modelo não parecer bom demais (ADR-0001 §5).

## Fase 2 — Data Understanding

`generate_synthetic_data` sorteia o AuC por segmento a partir de uma lognormal calibrada (`conf/base/parameters.yml`), com piso de R$ 3 M (mínimo de private banking, ANBIMA). `attach_advisor_and_behavioral_features` liga cada relação a um assessor e gera os 3 sinais de early-warning + o red-herring.

In [2]:
df = generate_synthetic_data(parameters["n_samples"], parameters=parameters, seed=SEED)
df_adv = generate_advisors_data(parameters["n_advisors"], seed=SEED)
df_v2, df_adv = attach_advisor_and_behavioral_features(df, df_adv, seed=SEED)

total_bi = df_v2["auc_milhoes"].sum() / 1000
vc = df_v2["churn"].value_counts()
print("AUDITORIA DE ESCALA (criterio de aceitacao ADR-0002 §6)")
print(f"  Relacoes            : {len(df_v2)}")
print(f"  AuC total           : R$ {total_bi:.1f} bi   (alvo 65-85)")
print(f"  AuC mediano         : R$ {df_v2['auc_milhoes'].median():.1f} M")
print(f"  Churn               : {vc.get(1,0)} ({vc.get(1,0)/len(df_v2)*100:.1f}%)")
print()
med = df_v2.groupby("segmento")["auc_milhoes"].agg(["count","median","sum"])
med["sum_bi"] = (med["sum"]/1000).round(2)
med["churn_%"] = (df_v2.groupby("segmento")["churn"].mean()*100).round(1)
print(med[["count","median","sum_bi","churn_%"]].to_string())
print()
ordenado = med["median"].reindex(["Alta Renda","Private","Wealth","Family Office"])
print("Mediana de AuC cresce Alta Renda < Private < Wealth < Family Office?",
      bool(ordenado.is_monotonic_increasing))

AUDITORIA DE ESCALA (criterio de aceitacao ADR-0002 §6)
  Relacoes            : 1200
  AuC total           : R$ 76.0 bi   (alvo 65-85)
  AuC mediano         : R$ 16.8 M
  Churn               : 140 (11.7%)

               count    median  sum_bi  churn_%
segmento                                       
Alta Renda       539    7.1870    4.47     14.7
Family Office     50  544.6300   34.65      6.0
Private          447   28.2590   14.94     11.2
Wealth           164  108.5945   21.99      4.9

Mediana de AuC cresce Alta Renda < Private < Wealth < Family Office? True


In [3]:
# t-Student de Welch: cada sinal comportamental separa quem fica de quem sai?
sinais = ["dias_desde_ultimo_contato", "variacao_freq_contato_3m",
          "tempo_resposta_medio_horas", "freq_contato_mes",
          "qtd_emails_marketing_recebidos"]  # o ultimo e o red-herring
g0 = df_v2[df_v2["churn"] == 0]
g1 = df_v2[df_v2["churn"] == 1]
print(f"{'feature':<32}{'media fica':>12}{'media sai':>12}{'p-value':>10}  sig?")
print("-"*70)
for f in sinais:
    a, b = g0[f].dropna(), g1[f].dropna()
    _, p = stats.ttest_ind(a, b, equal_var=False)
    print(f"{f:<32}{a.mean():>12.3f}{b.mean():>12.3f}{p:>10.4f}  {'[SIM]' if p<0.05 else '[NAO]'}")
print("\nEsperado: os 3 sinais de early-warning separam; qtd_emails_marketing (red-herring) nao.")

feature                           media fica   media sai   p-value  sig?
----------------------------------------------------------------------
dias_desde_ultimo_contato             15.599      22.218    0.0000  [SIM]
variacao_freq_contato_3m              -0.006      -0.145    0.0000  [SIM]
tempo_resposta_medio_horas            15.826      21.924    0.0000  [SIM]
freq_contato_mes                       2.794       2.664    0.4042  [NAO]
qtd_emails_marketing_recebidos         4.606       4.379    0.2595  [NAO]

Esperado: os 3 sinais de early-warning separam; qtd_emails_marketing (red-herring) nao.


## Fase 3 — Data Preparation

`inject_data_quality_issues` suja o dataset (duplicatas, nulos estruturais, sentinela `-999`, erro de escala) e `clean_clientes_v2_bruto` limpa — o pipeline aprende a lidar com dado real, não com dado de laboratório. O split é **estratificado e vem antes** da engenharia de features; qualquer transformador é ajustado só no treino (anti-leakage).

In [4]:
df_v2_bruto, df_adv_bruto = inject_data_quality_issues(df_v2, df_adv, seed=SEED)
df_v2_limpo = clean_clientes_v2_bruto(df_v2_bruto, df_adv_bruto)
print(f"Bruto  : {df_v2_bruto.shape}  | nulos: {int(df_v2_bruto.isna().sum().sum())} celulas")
print(f"Limpo  : {df_v2_limpo.shape}  | nulos estruturais preservados: "
      f"{int(df_v2_limpo[['retorno_12m_pct']].isna().sum().sum())} (sem historico de 12m)")

train_df, test_df = split_data(df, test_size=parameters["test_size"], random_state=SEED)
print(f"\nSplit estratificado — treino {len(train_df)} / teste {len(test_df)} "
      f"| churn no teste {test_df['churn'].sum()} ({test_df['churn'].mean()*100:.1f}%)")

df_fe, fe_params = run_feature_engineering(df, train_df)
print(f"\nFeature engineering (fit so no treino): media_retorno={fe_params['media_retorno']:.3f}")
print("  5 features derivadas para EDA/baseline:", [c for c in df_fe.columns
      if c not in df.columns and c != 'churn'])

Bruto  : (1218, 16)  | nulos: 175 celulas
Limpo  : (1200, 18)  | nulos estruturais preservados: 37 (sem historico de 12m)

Split estratificado — treino 960 / teste 240 | churn no teste 28 (11.7%)

Feature engineering (fit so no treino): media_retorno=11.616
  5 features derivadas para EDA/baseline: ['engajamento_score', 'retorno_relativo', 'flag_risco', 'intensidade_rel', 'segmento_enc']


## Fase 4 — Modeling

`train_and_compare_v1_v2` treina duas famílias sobre o **mesmo split** (mesma população, mesmo grão — gate ML de comparação):

- **v1 reativa:** só segmento, tempo, retorno, frequência, AuC — o modelo "antigo".
- **v2 early-warning:** o mesmo + os 3 sinais comportamentais + flags de nulo estrutural.

`benchmark_algorithms` dá o contexto de 5 algoritmos crescentes em complexidade.

In [5]:
comparacao, y_test_comum, y_pred_v1, y_pred_v2 = train_and_compare_v1_v2(df, df_v2_limpo, parameters)
print(comparacao.to_string(index=False))
print(f"\nn_teste={len(y_test_comum)} | eventos de churn no teste={int(y_test_comum.sum())}")

                  modelo  recall_churn  f1_churn  roc_auc  n_teste
     v1_baseline_reativa      0.000000  0.000000 0.472877      240
v2_early_warning_advisor      0.071429  0.114286 0.720856      240

n_teste=240 | eventos de churn no teste=28


In [6]:
bench = benchmark_algorithms(train_df, test_df, parameters)
print("Benchmark (baseline v1, para contexto de dificuldade do problema):")
print(bench[["modelo","f1_macro","f1_churn","roc_auc"]].to_string(index=False))

Benchmark (baseline v1, para contexto de dificuldade do problema):
             modelo  f1_macro  f1_churn  roc_auc
   Dummy (baseline)  0.512256  0.140351 0.512466
Logistic Regression  0.420913  0.201342 0.555930
      Decision Tree  0.356545  0.235897 0.587601
      Random Forest  0.467849  0.000000 0.452325
  Gradient Boosting  0.463087  0.000000 0.472877


## Fase 5 — Evaluation

Um único split de 240 relações é instável. `cross_validate_v2` roda 5 folds; `bootstrap_ic_diferenca_recall` dá o IC 95% da diferença de recall v2 − v1. Com ~28 eventos de churn no teste, o IC **inclui zero** — a v2 supera a v1 no ponto, mas o experimento ainda não prova ganho robusto. `calibrate_thresholds_v2` escolhe o corte por segmento numa curva de custo assimétrico (FN pesa 10× FP), em validação interna, nunca no teste.

In [7]:
ic = bootstrap_ic_diferenca_recall(y_test_comum, y_pred_v1, y_pred_v2, n_bootstrap=1000, seed=SEED)
cv_scores, cv_mean, cv_std = cross_validate_v2(df_v2_limpo, parameters)
print(f"Diferenca de recall (v2 - v1): {ic['diferenca_media']:+.4f}  "
      f"IC95% [{ic['ic95_lo']:+.4f}, {ic['ic95_hi']:+.4f}]")
print("  ->", "IC exclui zero" if ic["ic_exclui_zero"] else "IC INCLUI ZERO — pode ser ruido amostral")
print(f"\nCV 5-fold v2 — recall churn: {cv_mean:.4f} +/- {cv_std:.4f}")
print(cv_scores.to_string(index=False))

Diferenca de recall (v2 - v1): +0.0721  IC95% [+0.0000, +0.1787]
  -> IC INCLUI ZERO — pode ser ruido amostral

CV 5-fold v2 — recall churn: 0.1714 +/- 0.0416
  fold  recall_churn
Fold 1        0.1786
Fold 2        0.1071
Fold 3        0.2143
Fold 4        0.2143
Fold 5        0.1429


In [8]:
gb_v2 = train_final_model_v2(df_v2_limpo, parameters)
imp = get_feature_importance_v2(gb_v2)
comport = ["variacao_freq_contato_3m","dias_desde_ultimo_contato","tempo_resposta_medio_horas"]
print("Feature importance v2 (impurity-based):")
print(imp.to_string(index=False))
print(f"\nSoma das 3 features de early-warning: {imp[imp['feature'].isin(comport)]['importance'].sum():.1%}")

from sklearn.model_selection import train_test_split
cal_tr, cal_va = train_test_split(df_v2_limpo, test_size=0.25, random_state=SEED,
                                  stratify=df_v2_limpo["churn"])
thr = calibrate_thresholds_v2(train_final_model_v2(cal_tr, parameters), cal_va, parameters)
print("\nThresholds por segmento (custo 10*FN + FP, recall alvo 0,75):")
print(thr.to_string(index=False))

Feature importance v2 (impurity-based):
                      feature  importance
     variacao_freq_contato_3m    0.253912
    dias_desde_ultimo_contato    0.199148
   tempo_resposta_medio_horas    0.179410
                  auc_milhoes    0.085917
              retorno_12m_pct    0.057068
                meses_cliente    0.053332
             retorno_relativo    0.048743
            engajamento_score    0.035764
              intensidade_rel    0.035701
                 qtd_produtos    0.026460
             freq_contato_mes    0.014496
                 segmento_enc    0.009899
            sem_historico_12m    0.000133
                   flag_risco    0.000015
cliente_novo_sem_contato_hist    0.000000

Soma das 3 features de early-warning: 63.2%



Thresholds por segmento (custo 10*FN + FP, recall alvo 0,75):
     segmento  threshold   recall  precision  fn  fp  n_valid origem_threshold
   Alta Renda       0.10 0.500000       0.25  10  30      136         segmento
Family Office       0.10 0.000000       0.00   1   0       13  global_fallback
      Private       0.07 0.818182       0.25   2  27      111         segmento
       Wealth       0.10 0.666667       0.40   1   3       40  global_fallback


In [9]:
# Direcao B — produto de dado separado (nao entra no classificador)
carteira = aggregate_carteira_exposta_por_assessor(df_v2, df_adv)
exp_bi = carteira["auc_exposto_total"].sum() / 1000
print(f"AuC exposto agregado: R$ {exp_bi:.2f} bi "
      f"({carteira['auc_exposto_total'].sum()/df_v2['auc_milhoes'].sum()*100:.1f}% do total)")
print(f"pct_carteira_exposta — valores distintos: {carteira['pct_carteira_exposta'].nunique()} "
      f"(nao-degenerada), desvio {carteira['pct_carteira_exposta'].std():.3f}")
print(carteira.sort_values('auc_exposto_total', ascending=False).head(5).to_string(index=False))

AuC exposto agregado: R$ 9.30 bi (12.2% do total)
pct_carteira_exposta — valores distintos: 49 (nao-degenerada), desvio 0.041
assessor_id  qtd_clientes  auc_total_carteira  auc_exposto_total  pct_carteira_exposta         canal  anos_de_casa  risco_saida
    ADV0048            39           20207.986           2429.001                0.1202 Broker-Dealer          13.7            0
    ADV0049            25           15941.040           2228.559                0.1398     Wirehouse          10.8            0
    ADV0045            69            8912.458            763.798                0.0857 Broker-Dealer          26.8            0
    ADV0047            30            4611.499            557.072                0.1208 Broker-Dealer           8.1            0
    ADV0044            19            2731.916            441.477                0.1616     Wirehouse           6.8            0


## Fase 6 — Deployment

O artefato de produção é `output/models/gb_pipeline_v2.pkl`, gerado por `python pipeline.py` — este notebook **não** o regrava. Abaixo, o mesmo scoring que `api.py` faz: dados brutos entram, o `Pipeline` serializado cuida da imputação de nulo estrutural, FE e encoding, e devolve a probabilidade. A classificação usa o threshold calibrado do segmento.

In [10]:
import joblib
PKL = "../output/models/gb_pipeline_v2.pkl"
THR = "../output/data/thresholds_v2.csv"
if not os.path.exists(PKL):
    raise SystemExit("Rode `python pipeline.py` na raiz do projeto antes desta celula.")

modelo_prod = joblib.load(PKL)
thr_map = pd.read_csv(THR).set_index("segmento")["threshold"].to_dict()
FEATURES_V2 = ["segmento","meses_cliente","qtd_produtos","retorno_12m_pct","freq_contato_mes",
               "auc_milhoes","dias_desde_ultimo_contato","variacao_freq_contato_3m",
               "tempo_resposta_medio_horas","sem_historico_12m","cliente_novo_sem_contato_hist"]

def score(nome, **campos):
    campos.setdefault("retorno_12m_pct", None)
    campos.setdefault("dias_desde_ultimo_contato", None)
    campos.setdefault("tempo_resposta_medio_horas", 16.0)
    campos["sem_historico_12m"] = int(campos["retorno_12m_pct"] is None)
    campos["cliente_novo_sem_contato_hist"] = int(campos["dias_desde_ultimo_contato"] is None)
    X = pd.DataFrame([campos])[FEATURES_V2]
    p = float(modelo_prod.predict_proba(X)[0][1])
    t = thr_map.get(campos["segmento"], 0.5)
    nivel = "ALTO" if p >= t else "MEDIO" if p >= t*0.6 else "BAIXO"
    print(f"{nome:<26} prob={p*100:5.1f}%  corte {t*100:.0f}%  -> {nivel}")
    return p

print("SIMULACAO DE SCORING (schema v2 early-warning)\n" + "-"*60)
score("Private, deteriorando",  segmento="Private", meses_cliente=30, qtd_produtos=3,
      freq_contato_mes=0, auc_milhoes=25.0, retorno_12m_pct=6.0,
      dias_desde_ultimo_contato=70.0, variacao_freq_contato_3m=-0.45, tempo_resposta_medio_horas=60.0)
score("Alta Renda, cadencia caindo", segmento="Alta Renda", meses_cliente=24, qtd_produtos=3,
      freq_contato_mes=1, auc_milhoes=8.0, retorno_12m_pct=9.0,
      dias_desde_ultimo_contato=30.0, variacao_freq_contato_3m=-0.15, tempo_resposta_medio_horas=24.0)
score("Wealth, relacionamento saudavel", segmento="Wealth", meses_cliente=96, qtd_produtos=6,
      freq_contato_mes=5, auc_milhoes=150.0, retorno_12m_pct=14.0,
      dias_desde_ultimo_contato=6.0, variacao_freq_contato_3m=0.05, tempo_resposta_medio_horas=8.0)

SIMULACAO DE SCORING (schema v2 early-warning)
------------------------------------------------------------
Private, deteriorando      prob= 77.1%  corte 7%  -> ALTO
Alta Renda, cadencia caindo prob= 28.7%  corte 10%  -> ALTO
Wealth, relacionamento saudavel prob=  0.6%  corte 10%  -> BAIXO


0.005962218574588542

## O que este notebook demonstra

Contrato de dados temporal, separação de dois produtos analíticos, engenharia de features sem *leakage*, comparação modelo-vs-baseline na mesma população, IC bootstrap sobre amostra pequena e calibração de threshold por custo. A v2 supera a v1 em recall no mesmo split, mas com ~28 eventos de churn no teste o resultado **não** é um ganho robusto, e o dado é sintético — nenhum número aqui é efeito de negócio observado.

Próximo: `python pipeline.py` para os artefatos versionados, `uvicorn api:app` para o serviço, `streamlit run app.py` para o dashboard.